## Section-1: Setup and Dependency Installation
In this cell, we install the necessary Python packages required for the project, keeping output quiet.


In [ ]:
!pip install scipy numpy matplotlib torch torchvision \
    torch-geometric umap-learn wandb networkx tqdm -q

## Repository Setup
Here, we clone the project repository from GitHub to the Colab environment to access the source code.


In [ ]:
import os
REPO_ROOT = '/content/antenna-gnn'
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/asparagusD/antenna_gnn.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
import sys
sys.path.insert(0, f'{REPO_ROOT}/src')
print(f'Repo ready at {REPO_ROOT}')

## Drive Mount and Path Configuration
This cell mounts Google Drive, sets up root paths, ensures necessary directories exist, and verifies GPU availability.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
RAW_DATA  = '/content/drive/MyDrive/antenna_dataset'
for d in [f'{DATA_ROOT}/artifacts', f'{DATA_ROOT}/checkpoints',
          f'{DATA_ROOT}/figures',   f'{DATA_ROOT}/splits',
          f'{DATA_ROOT}/data/processed']:
    os.makedirs(d, exist_ok=True)
print(f'Drive mounted. DATA_ROOT={DATA_ROOT}')

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'Enable GPU first'

## Section 2: Load dataset
We use `glob` to locate the 25x25 training dataset files and load all files. We treat the patch pattern as a single-channel 2D image. Finally, we apply an 80/10/10 data split and save the indices.


In [ ]:
import glob
import scipy.io as sio
import numpy as np
from tqdm import tqdm
import json
from sklearn.model_selection import StratifiedShuffleSplit

all_files = sorted(glob.glob(
    f'{RAW_DATA}/training dataset/25x25/**/Mat_Files/*.mat',
    recursive=True))

print(f"Total files found: {len(all_files)}")

X_list = []
y_list = []
is_functioning_list = []

for i, path in enumerate(tqdm(all_files, desc="Loading dataset")):
    mat = sio.loadmat(path)
    X_row = torch.tensor(mat['patch_pattern'], dtype=torch.float32).unsqueeze(0)
    y_row = torch.tensor(mat['S11_dB'].flatten(), dtype=torch.float32)
    is_functioning = int(mat['resonant_freqs'].size > 0)
    
    X_list.append(X_row)
    y_list.append(y_row)
    is_functioning_list.append(is_functioning)
    
    if (i + 1) % 10000 == 0:
        print(f"Loaded {i + 1} files...")

X = torch.stack(X_list)
y = torch.stack(y_list)
y_func = np.array(is_functioning_list)

# Split 80/10/10 stratified by is_functioning
sss_test = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_val_idx, test_idx = next(sss_test.split(np.zeros(len(y_func)), y_func))

# Remaining 90% is split to 80% and 10% (test_size=1/9 of 90%)
sss_val = StratifiedShuffleSplit(n_splits=1, test_size=1/9, random_state=42)
train_idx_rel, val_idx_rel = next(sss_val.split(np.zeros(len(train_val_idx)), y_func[train_val_idx]))

train_idx = train_val_idx[train_idx_rel]
val_idx = train_val_idx[val_idx_rel]

with open(f'{DATA_ROOT}/splits/cnn_split_indices.json', 'w') as f:
    json.dump({
        'train': train_idx.tolist(),
        'val': val_idx.tolist(),
        'test': test_idx.tolist()
    }, f)

print(f"Total samples: {len(X)}")
print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")
print(f"Functioning % in Train: {100 * np.mean(y_func[train_idx]):.2f}%")
print(f"Functioning % in Val:   {100 * np.mean(y_func[val_idx]):.2f}%")
print(f"Functioning % in Test:  {100 * np.mean(y_func[test_idx]):.2f}%")

## Section 3: CNN model (Gupta et al. 2023, adapted)
We define the Convolutional Neural Network baseline. The architecture uses stacked convolutional layers followed by AdaptiveAvgPool2d, allowing it to dynamically handle any input grid size (NxN).


In [ ]:
import torch.nn as nn

class AntennaCNN(nn.Module):
    """
    Adapted from Gupta et al. IEEE TAP 2023, Fig. 4.
    Original: 12x12 input, 81 output points.
    Adapted: NxN input via AdaptiveAvgPool2d, 201 output points (1-4 GHz).
    """
    def __init__(self):
        super().__init__()
        
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=5, padding=2),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, kernel_size=5, padding=2),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, kernel_size=5, padding=2),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2)
        )
        
        layers2 = []
        layers2.extend([nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2)])
        for _ in range(4):
            layers2.extend([nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2)])
        self.block2 = nn.Sequential(*layers2)
        
        layers3 = []
        layers3.extend([nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2)])
        for _ in range(7):
            layers3.extend([nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2)])
        self.block3 = nn.Sequential(*layers3)
        
        self.readout = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )
        
        self.fc = nn.Sequential(
            nn.Linear(256, 1000),
            nn.BatchNorm1d(1000),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Linear(1000, 500),
            nn.BatchNorm1d(500),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Linear(500, 201)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.readout(x)
        x = self.fc(x)
        return x

model = AntennaCNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## Section 4: Training loop
We set up a robust training loop utilizing NAdam and MSE loss. The `save_checkpoint` pattern prevents lost progress by resuming from the most recent checkpoint. Memory is actively managed by explicitly sending batches to the device and deleting them afterward.


In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_loader = DataLoader(TensorDataset(X[train_idx], y[train_idx]), batch_size=256, shuffle=True)
val_loader = DataLoader(TensorDataset(X[val_idx], y[val_idx]), batch_size=256)

optimizer = torch.optim.NAdam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

def save_checkpoint(model, optimizer, epoch, val_loss, path):
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'val_loss': val_loss,
    }, path)

CKPT_PATH = f'{DATA_ROOT}/checkpoints/cnn_best.pt'
start_epoch = 0
best_val_loss = float('inf')

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt['val_loss']
    print(f'Resumed from epoch {start_epoch}, val_loss={best_val_loss:.6f}')
else:
    print('Starting fresh training')

train_losses = []
val_losses = []
epochs = 40

for epoch in range(start_epoch, epochs):
    model.train()
    total_train_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        out = model(batch_X)
        loss = criterion(out, batch_y)
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item() * batch_X.size(0)
        del batch_X, batch_y
        torch.cuda.empty_cache()
        
    avg_train_loss = total_train_loss / len(train_idx)
    
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            out = model(batch_X)
            loss = criterion(out, batch_y)
            total_val_loss += loss.item() * batch_X.size(0)
            del batch_X, batch_y
            torch.cuda.empty_cache()
            
    avg_val_loss = total_val_loss / len(val_idx)
    
    # Store for plotting (if resuming, this list only tracks the new epochs)
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        save_checkpoint(model, optimizer, epoch, best_val_loss, CKPT_PATH)
        
    if (epoch + 1) % 10 == 0 or epoch == start_epoch:
        print(f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

## Section 5: Training curves
We visualize the training trajectory. Note that since our training subset has ~100k samples, our final loss may be significantly higher than the 0.62 (train) and 0.71 (val) reported in Gupta et al. (trained on 500,000).


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if train_losses:
    plt.figure(figsize=(10, 6))
    plt.plot(range(start_epoch, epochs), train_losses, label='Train Loss')
    plt.plot(range(start_epoch, epochs), val_losses, label='Val Loss')
    plt.title('CNN Baseline Training Curves (Gupta et al. 2023 architecture)')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(f'{DATA_ROOT}/figures/cnn_training_curves.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Final Train Loss: {train_losses[-1]:.4f}")
    print(f"Final Val Loss: {val_losses[-1]:.4f}")
else:
    print("Model was already fully trained.")

## Section 6: Evaluation
Here, we load the best checkpoint and evaluate it against our 10% test split. We compute the MAE over the whole S11 spectrum, as well as the MAE of the critical extracted resonant frequency (for antennas exhibiting a resonance).


In [ ]:
from scipy.signal import find_peaks

def extract_resonant_freq(s11_db, freq_axis_ghz, threshold_db=-10):
    inverted = -s11_db
    peaks, props = find_peaks(inverted, height=-threshold_db, distance=5)
    if len(peaks) == 0:
        return None
    deepest = peaks[np.argmax(inverted[peaks])]
    return freq_axis_ghz[deepest]

ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()

freq_axis = np.linspace(1.0, 4.0, 201)
test_loader = DataLoader(TensorDataset(X[test_idx], y[test_idx]), batch_size=256)

all_preds = []
all_trues = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        out = model(batch_X.to(device))
        all_preds.append(out.cpu())
        all_trues.append(batch_y)

all_preds = torch.cat(all_preds).numpy()
all_trues = torch.cat(all_trues).numpy()

mae_s11 = np.mean(np.abs(all_preds - all_trues))

mae_freqs = []
for i in range(len(all_trues)):
    true_res = extract_resonant_freq(all_trues[i], freq_axis)
    if true_res is not None:
        pred_res = extract_resonant_freq(all_preds[i], freq_axis)
        if pred_res is not None:
            mae_freqs.append(abs(true_res - pred_res))

mae_freq = np.mean(mae_freqs) if len(mae_freqs) > 0 else float('nan')

print("--------------------------------------------------")
print("=== Test Set Evaluation ===")
print(f"MAE S11 Spectrum : {mae_s11:.4f} dB")
print(f"MAE Resonant Freq: {mae_freq:.4f} GHz")
print("--------------------------------------------------")

## Section 7: Prediction plots
We visualize a selection of 6 test antennas (3 functioning, 3 non-functioning) by comparing the true simulated S11 curve (blue) to the CNN predicted curve (red dashed).


In [ ]:
import random

functioning_idx = []
non_functioning_idx = []

for i in range(len(all_trues)):
    if extract_resonant_freq(all_trues[i], freq_axis) is not None:
        functioning_idx.append(i)
    else:
        non_functioning_idx.append(i)

random.seed(42)
sampled_func = random.sample(functioning_idx, min(3, len(functioning_idx)))
sampled_nonfunc = random.sample(non_functioning_idx, min(3, len(non_functioning_idx)))
plot_indices = sampled_func + sampled_nonfunc

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, idx in enumerate(plot_indices):
    ax = axes[i]
    ax.plot(freq_axis, all_trues[idx], 'b-', label='True')
    ax.plot(freq_axis, all_preds[idx], 'r--', label='Predicted')
    ax.axhline(-10, color='gray', linestyle=':')
    title = 'Functioning' if i < len(sampled_func) else 'Non-functioning'
    ax.set_title(title)
    ax.set_xlabel('Frequency (GHz)')
    ax.set_ylabel('S11 (dB)')
    if i == 0:
        ax.legend()

plt.tight_layout()
plt.savefig(f'{DATA_ROOT}/figures/cnn_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

## Section 8: Permutation sensitivity demonstration
The CNN processes the antenna as a rigid 2D image, inherently treating spatially permuted copies (like flips or transposes) as mathematically distinct objects. Consequently, the network produces varying predictions for the exact same physical antenna structure simply drawn in a different orientation.

In contrast, the Graph Neural Network framework inherently treats pixels as graph nodes. The structural topology is completely independent of visual orientation, rendering it structurally robust to these spatial permutations.


In [ ]:
random.seed(42)
perm_test_indices = random.sample(range(len(test_idx)), 20)
test_X = X[test_idx]

std_list = []
example_preds = []

model.eval()
with torch.no_grad():
    for i, idx in enumerate(perm_test_indices):
        orig = test_X[idx] # (1, 25, 25)
        orig_t = torch.transpose(orig, 1, 2)
        
        variants = [
            orig,
            orig_t,
            torch.flip(orig, [2]),
            torch.flip(orig, [1]),
            torch.flip(orig_t, [2]),
            torch.flip(orig_t, [1]),
            torch.flip(orig, [1, 2]),
            torch.flip(orig_t, [1, 2])
        ]
        
        batch_var = torch.stack(variants).to(device) # (8, 1, 25, 25)
        preds = model(batch_var).cpu().numpy() # (8, 201)
        
        std_list.append(np.std(preds, axis=0)) # (201,) std across variants
        
        if i == 0:
            example_preds = preds

mean_sensitivity = np.mean(std_list)
print(f"CNN permutation sensitivity: {mean_sensitivity:.2f} dB")

plt.figure(figsize=(8, 6))
for i in range(8):
    plt.plot(freq_axis, example_preds[i], label=f'Variant {i+1}')
plt.axhline(-10, color='gray', linestyle=':')
plt.title('CNN Predictions on 8 Spatial Variants of 1 Antenna')
plt.xlabel('Frequency (GHz)')
plt.ylabel('S11 (dB)')
plt.legend()
plt.savefig(f'{DATA_ROOT}/figures/cnn_permutation_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()

CNN now trained on the full 25x25 dataset (N=XXXXX samples) to match the GNN's training data volume for a fair architecture comparison in Chunk 8.